<a href="https://colab.research.google.com/github/safaltasaxena/deep-learning-mini-projects/blob/main/tf_preFetch_cache/tf_pp_opt(prefetch%2Ccache).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

measuring the performance using prefetch

In [ ]:
import tensorflow as tf
import time

In [ ]:
tf.__version__

'2.20.0'

In [ ]:
import tensorflow as tf
import time

#mimic latency
class FileDataSet:
   @staticmethod
   def read_files_in_bacthes(num_samples):
    #open file
    time.sleep(0.03)
    for sample_idx in range(num_samples):
      time.sleep(0.015)
      yield (sample_idx,)
   def __new__(cls,num_samples=3):
      return tf.data.Dataset.from_generator(
          cls.read_files_in_bacthes,
          output_signature=tf.TensorSpec(shape=(1,), dtype=tf.int64),
          args=(num_samples,)
      )

In [ ]:
def benchmark(dataset,num_epochs=2):
  for epoch_num in range(num_epochs):
    for sample in dataset:
      time.sleep(0.01)

In [ ]:
%%timeit
benchmark(FileDataSet())

259 ms ± 5.32 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [ ]:
%%timeit
benchmark(FileDataSet().prefetch(1))

251 ms ± 869 µs per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [ ]:
%%timeit
benchmark(FileDataSet().prefetch(tf.data.AUTOTUNE))

259 ms ± 6.31 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


Cache optimise

In [ ]:
dataset=tf.data.Dataset.range(5)
for d in dataset:
  print(d.numpy())

0
1
2
3
4


In [ ]:
dataset=dataset.map(lambda x:x**2)
for d in dataset:
  print(d.numpy())

0
1
4
9
16


In [ ]:
dataset=dataset.cache()
list(dataset.as_numpy_iterator())

[np.int64(0), np.int64(1), np.int64(4), np.int64(9), np.int64(16)]

In [ ]:
list(dataset.as_numpy_iterator())

[np.int64(0), np.int64(1), np.int64(4), np.int64(9), np.int64(16)]

In [ ]:
def mapped_fnc(S):
  # S is a tuple (tensor,) because FileDataSet yields (sample_idx,)
  # We need to operate on the tensor itself, which is S[0]

  def _py_fnc(val_tensor): # val_tensor is a tf.Tensor
    import time # Ensure time is available in the py_function's scope
    time.sleep(0.03)
    return val_tensor.numpy() # Return a Python scalar

  # Wrap the Python function in tf.py_function
  # Since Tout is a list [tf.int64], tf.py_function returns a list containing one tensor.
  output_list = tf.py_function(
      func=_py_fnc,
      inp=[S[0]], # Pass the actual tensor from the tuple (a scalar tensor)
      Tout=[tf.int64] # Expecting an int64 tensor back
  )
  # Extract the actual tensor from the list returned by tf.py_function
  output_tensor = output_list[0]

  # Ensure the shape of the output_tensor is correctly set as scalar
  output_tensor.set_shape([])

  # The dataset expects elements as tuples, so return a tuple with the processed tensor
  return (output_tensor,)

In [ ]:
%%timeit -n1 -r7
benchmark(FileDataSet().map(mapped_fnc),5)

1.13 s ± 32.4 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [ ]:
%%timeit -n1 -r7
benchmark(FileDataSet().map(mapped_fnc).cache(),5)

397 ms ± 17.3 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
